<a href="https://colab.research.google.com/github/AsiyaKondusa/KnowledgeEngineering/blob/main/Autonomous_Trailer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Knowledge Based Engineering: Group 12 Notebook

## Step 1 - Identify the task

- Develop self-driving trailer agent as a knowldege-based agentto make safe driving decision with minimal risk.
- Helping the self-driving trailer get from (Nairobi  → Kisumu)
- Actions to do (stop, slow down, go) in the course of the route to the destination.



## Step 2: Assemble relevant Knowledge

Things to know:
- Towns and roads between them(distances in km)
- Traffic Light color
- Action to take based on colour
- Traffic laws and road conditions (current weather conditions)
- Blocked roads

API declaration:
1. Open-Meteo weather API - It is used to get the current rainfall at a town(free with no key needed)
https://api.open-meteo.com/v1/forecast
Terms: https://open-meteo.com/en/terms

2. OpenStreetMap Nominatim: gets the latitude/longitude of each town

   https://nominatim.openstreetmap.org/search
   
   Usage policy: https://operations.osmfoundation.org/policies/nominatim/


In [40]:
# Imports for all the cells
import requests # lets Python ask a website / API for data
import time # Lets us pause btn requests

In [41]:
# Nominatim's policy
HEADERS = {"User-Agent": "Knowledge_Based_Agent_Trailer/1.0 (student project; amani.koros@strathmore.edu)"}

TOWNS = ["Nairobi", "Naivasha", "Narok", "Nakuru", "Bomet", "Litein",
         "Kisii", "Eldoret", "Kericho", "HomaBay", "Kisumu"]
SEARCH_NAME = {"HomaBay": "Homa Bay"}          # OpenStreetMap spells this one with a space

def get_coordinates(town):
    # Ask OpenStreetMap (Nominatim) for the (latitude, longitude) of a town in Kenya.
    params = {"q": f"{SEARCH_NAME.get(town, town)}, Kenya", "format": "json", "limit": 1,
              "countrycodes": "ke",             # only look inside Kenya
              "featuretype": "settlement"}      # only towns/cities/villages, not counties or wards
    try:
        r = requests.get("https://nominatim.openstreetmap.org/search",
                         params=params, headers=HEADERS, timeout=10)
        first = r.json()[0]                                              # best match
        return float(first["lat"]), float(first["lon"]), first["type"]   # lat/lon arrive as text, so convert
    except Exception:
        return None                                                      # no internet or town not found

# Look up every town once and keep the answers in dictionaries
TOWN_COORDS = {}
TOWN_TYPE = {}
for town in TOWNS:
    result = get_coordinates(town)
    if result:
        lat, lon, kind = result
        TOWN_COORDS[town] = (lat, lon)
        TOWN_TYPE[town] = kind                                # what OpenStreetMap says it found (town, city, village...)
    time.sleep(1)                                             # respect the 1 request per second rule

# Town-centre coordinates checked by hand on openstreetmap.org (Narok, Bomet, Homa Bay, were not dipslaying the correct values using the openstreeetmap api, hence we have inputted the values manually)
VERIFIED = {"Narok": (-1.08447, 35.87173),
            "Bomet": (-0.77843, 35.34370),
            "HomaBay": (-0.5684, 34.4758)}
TOWN_COORDS.update(VERIFIED)

print("Coordinates from OpenStreetMap:")
for town, (lat, lon) in TOWN_COORDS.items():
    print(f"  {town:9} {lat:.4f}, {lon:.4f}  ({TOWN_TYPE[town]})")

def get_weather(town):
    # Ask Open-Meteo how much rain (mm) is falling at a town right now, and turn it into a simple label.
    if town not in TOWN_COORDS:
        return None                                           # if they are no coordinates for this town
    lat, lon = TOWN_COORDS[town]
    url = "https://api.open-meteo.com/v1/forecast"
    params = {"latitude": lat, "longitude": lon, "current": "precipitation"}
    try:
        data = requests.get(url, params=params, timeout=10).json()   # send the request, read the answer
        rain_mm = data["current"]["precipitation"]                   # pick out the rainfall measurement in mm
    except Exception:
        return None                                                  # no internet or the API is down
    if rain_mm >= 5:
        return "HeavyRain"
    if rain_mm > 0:
        return "Rain"
    return "Clear"

print("Weather in Nakuru right now:", get_weather("Nakuru"))

Coordinates from OpenStreetMap:
  Nairobi   -1.2890, 36.8173  (city)
  Naivasha  -0.7224, 36.4388  (town)
  Narok     -1.0845, 35.8717  (administrative)
  Nakuru    -0.2803, 36.0712  (city)
  Bomet     -0.7784, 35.3437  (administrative)
  Litein    -0.5826, 35.1881  (town)
  Kisii     -0.7389, 34.7540  (administrative)
  Eldoret   0.5198, 35.2715  (city)
  Kericho   -0.3210, 35.2261  (administrative)
  HomaBay   -0.5684, 34.4758  (administrative)
  Kisumu    -0.1029, 34.7542  (city)
Weather in Nakuru right now: Clear


## Step 3: Defined Vocabulary

1.  Predicates: Keys and rules represented in Python
- Road(a,b)km,
- Blocked(a,b),
- Light(colour),
- RoadType(a,b),
- SpeedLimit(type),
- Condition(town,c)

2. Constants: All towns in the routes.
- { Nairobi → Kisumu};
- Lights {Red, Orange, Green};
- Conditions(Clear, Rain, HeavyRain)

## Step 4: Encoding Knowledge about the Domain

- Predicates and Constants defined in Step 3, are now written as Python Dictionaries.

- Each Python dictionary is 1 kind of knowldege

In [42]:
# routes from the directed graph from UCS_trailer
roads = {
    "Nairobi": {"Naivasha": 91, "Narok": 143},
    "Naivasha": {"Nakuru": 71},
    "Narok": {"Bomet": 81},
    "Nakuru": {"Eldoret": 156, "Kericho": 107},
    "Bomet": {"Litein": 44.1,"Kisii": 82.4},
    "Eldoret":{"Kisumu": 138},
    "Kericho":{"Kisumu": 84},
    "Litein":{"Kericho": 32},
    "Kisii":{"HomaBay": 52},
    "HomaBay":{"Kisumu": 108},
    "Kisumu":{},
}

# Light(colour)
light_rules = {
    "Red": "Stop, it is unsafe to move.",
    "Yellow": "Slow down and prepare to stop.",
    "Green": "Go: it is safe to proceed."
}

# Condition(c) -> weather / road condition
condition_rules = {
    "Clear": "Drive safely.",
    "Rain": "Slow down and keep a longer distance between the car infront.",
    "HeavyRain": "Unsafe: stop and wait"
}

# Traffic law -- Speed limit of 80 km/h depending on the road type (Superhighway, Highway, Murram road, e.t.c..)
speed_limits = {"Highway": 80}
road_types = {("Nairobi", "Naivasha"): "Highway",
              ("Naivasha", "Nakuru"): "Highway"}

# Blocked(a,b) -> Empty at first then continued in Step 5
blocked_roads = set()

# Quick test: can the Knowledge based agent print some output:
print(light_rules["Green"])
print("Speed limit Nairobi-> Naivasha:", speed_limits[road_types[("Nairobi", "Naivasha")]], "km/h")

Go: it is safe to proceed.
Speed limit Nairobi-> Naivasha: 80 km/h


## Step 5: Add New Facts to Knowledge Base Agent

In [43]:
def show_kb():
    # Print everything the agent currently knows (apart from the road distances, which are in the graph).
    print("\n--- Current Knowledge Base ---")
    print("Light rules:")
    for light, action in light_rules.items():
        print(f"  {light:10} ➜ {action}")
    print("Condition rules:")
    for cond, action in condition_rules.items():
        print(f"  {cond:10} ➜ {action}")
    print("Blocked roads:", sorted(blocked_roads) if blocked_roads else "none")
    print("-------------------------------")

def add_new_fact(color, action):
    # Add a new traffic light fact directly to the KB (without user input).
    light_rules[color.capitalize()] = action
    print(f"Added fact: '{color.capitalize()}' ➜ '{action}'")

def block_road(a, b):
    # Add the fact Blocked(a, b) so the agent knows not to use that road.
    blocked_roads.add((a, b))
    print(f"Added fact: road {a} -> {b} is BLOCKED")

# Examples of adding facts
add_new_fact("Flashing", "Stop and check intersection")
block_road("Nakuru", "Eldoret")                                  # e.g. a road closed by flooding

# A fact that comes from the API: the current weather in Nakuru
weather = get_weather("Nakuru")
if weather:                                                      # only if the API answered
    print("Live weather in Nakuru:", weather, "->", condition_rules[weather])

def add_fact():
    # Let the user add a new traffic light colour and its action.
    new_light = input("Enter new traffic light color: ").capitalize()
    new_action = input(f"What should the agent do when light is {new_light}? ")
    light_rules[new_light] = new_action
    print(f" New fact added: '{new_light}' -> '{new_action}'")

add_fact()

# View updated KB
show_kb()

Added fact: 'Flashing' ➜ 'Stop and check intersection'
Added fact: road Nakuru -> Eldoret is BLOCKED
Live weather in Nakuru: Clear -> Drive safely.
Enter new traffic light color: Green
What should the agent do when light is Green? Go
 New fact added: 'Green' -> 'Go'

--- Current Knowledge Base ---
Light rules:
  Red        ➜ Stop, it is unsafe to move.
  Yellow     ➜ Slow down and prepare to stop.
  Green      ➜ Go
  Flashing   ➜ Stop and check intersection
Condition rules:
  Clear      ➜ Drive safely.
  Rain       ➜ Slow down and keep a longer distance between the car infront.
  HeavyRain  ➜ Unsafe: stop and wait
Blocked roads: [('Nakuru', 'Eldoret')]
-------------------------------


In [44]:
# STEP 6: SPECIFIC PROBLEM FACTS

# definition of the  current location and next destination
current_location = "Nairobi"
next_location = "Naivasha"

# current traffic-light colour
traffic_light = "Green"

# current weather condition
weather_condition = "Rain"

# current facts displayed
print("Current Location:", current_location)
print("Next Location:", next_location)
print("Traffic Light:", traffic_light)
print("Weather Condition:", weather_condition)

Current Location: Nairobi
Next Location: Naivasha
Traffic Light: Green
Weather Condition: Rain


In [45]:
# STEP 7: INFERENCE ENGINE

def inference_engine(current_location, next_location, light, weather):

    # make everything in a standard format
    light = light.strip().capitalize()
    weather = weather.strip().lower()

    # R1: If the road is blocked, find an alternative route
    if (current_location, next_location) in blocked_roads:
        return "FIND AN ALTERNATIVE ROUTE!!"

    # R2: A red traffic light = trailer must stop
    if light == "Red":
        return "STOP"

    # R3: traffic-light colour unknown = stop for safety
    if light not in light_rules:
        return "STOP - Unknown traffic light"

    # R3b: flashing light = stop and check the intersection
    if light == "Flashing":
        return "STOP - Check intersection"

    # R4: Heavy rain makes driving unsafe = trailer must stop
    if weather == "heavyrain":
        return "STOP - Heavy rain"

    # R5: Rain + green light = trailer to slowing down
    if weather == "rain" and light == "Green":
        return "SLOW DOWN"

    # R6: yellow traffic light = slow down & prepare to stop
    if light == "Yellow":
        return "SLOW DOWN"

    # R7: green traffic light + clear weather =  trailer to proceed
    if light == "Green" and weather == "clear":
        return "GO"

    # DEFAULT RULE: Stop if the conditions do not match any known rule
    return "STOP - Unknown conditions"

In [46]:
# STEP 8: TEST THE INFERENCE ENGINE

# Pass the current facts to the inference engine
decision = inference_engine(
    current_location,
    next_location,
    traffic_light,
    weather_condition
)

# Display the decision
print("Autonomous Trailer Decision:", decision)

Autonomous Trailer Decision: SLOW DOWN


In [47]:
# STEP 9: TEST DIFFERENT INFERENCE RULES

# Each test contains:
# Current location, next location, traffic light and weather condition

test_queries = [
    ("Nairobi", "Naivasha", "Red", "Clear"),
    ("Nairobi", "Naivasha", "Green", "HeavyRain"),
    ("Nairobi", "Naivasha", "Green", "Clear"),
    ("Nairobi", "Naivasha", "Yellow", "Clear"),
    ("Nairobi", "Naivasha", "Purple", "Clear"),
    ("Nairobi", "Naivasha", "Green", "Rain")
]

# Run each test throuh the inferenc engine
for current, next_town, light, weather in test_queries:

    result = inference_engine(current, next_town, light, weather)

    # inputs and the resulting decision
    print(
        f"Location: {current} -> {next_town} | "
        f"Light: {light} | Weather: {weather} | "
        f"Decision: {result}"
    )

Location: Nairobi -> Naivasha | Light: Red | Weather: Clear | Decision: STOP
Location: Nairobi -> Naivasha | Light: Green | Weather: HeavyRain | Decision: STOP - Heavy rain
Location: Nairobi -> Naivasha | Light: Green | Weather: Clear | Decision: GO
Location: Nairobi -> Naivasha | Light: Yellow | Weather: Clear | Decision: SLOW DOWN
Location: Nairobi -> Naivasha | Light: Purple | Weather: Clear | Decision: STOP - Unknown traffic light
Location: Nairobi -> Naivasha | Light: Green | Weather: Rain | Decision: SLOW DOWN


In [48]:
# STEP 10: TEST SUMMARY

# below are test cases containing:
# Traffic light, weather condition and expected decision

test_cases = [
    ("red", "clear", "STOP"),
    ("green", "heavyrain", "STOP - Heavy rain"),
    ("green", "clear", "GO"),
    ("yellow", "clear", "SLOW DOWN"),
    ("purple", "clear", "STOP - Unknown traffic light"),
    ("green", "rain", "SLOW DOWN"),
    ("flashing", "clear", "STOP - Check intersection")
]

# Run each test and compare the actual and expected results
for light, weather, expected in test_cases:

    actual = inference_engine(
        "Nairobi",
        "Naivasha",
        light,
        weather
    )

    # Check whether the test passed or failed
    status = "PASS" if actual == expected else "FAIL"

    print(
        f"Light -  {light} , weather- {weather} , "
        f"expected- {expected} , actual - {actual} , {status}"
    )

Light -  red , weather- clear , expected- STOP , actual - STOP , PASS
Light -  green , weather- heavyrain , expected- STOP - Heavy rain , actual - STOP - Heavy rain , PASS
Light -  green , weather- clear , expected- GO , actual - GO , PASS
Light -  yellow , weather- clear , expected- SLOW DOWN , actual - SLOW DOWN , PASS
Light -  purple , weather- clear , expected- STOP - Unknown traffic light , actual - STOP - Unknown traffic light , PASS
Light -  green , weather- rain , expected- SLOW DOWN , actual - SLOW DOWN , PASS
Light -  flashing , weather- clear , expected- STOP - Check intersection , actual - STOP - Check intersection , PASS


In [49]:
# STEP 11: TEST BLOCKED-ROAD DETECTION

# Temporarily add the Nairobi to Naivasha road to the blocked roads set
blocked_roads.add(("Nairobi", "Naivasha"))

# Test the inference engine with the blocked road
blocked_result = inference_engine(
    "Nairobi",
    "Naivasha",
    "Green",
    "Clear"
)

# Display the result
print("Blocked Road Test:", blocked_result)

# Remove the temporary blockage after testing
blocked_roads.discard(("Nairobi", "Naivasha"))

Blocked Road Test: FIND AN ALTERNATIVE ROUTE!!


In [50]:
# STEP 12: ALTERNATIVE ROUTE PLANNER USING UCS

import heapq

def alternative_route(start, destination, roads, blocked_roads):

    # doboth towns exist?
    if start not in roads or destination not in roads:
        return None, float("inf")

    # Priority queue - (distance, current town, route)
    queue = [(0, start, [start])]
    best_distance = {start: 0}

    while queue:

        # lowest distance route
        distance, current, path = heapq.heappop(queue)

        # Skip outdated routes
        if distance > best_distance.get(current, float("inf")):
            continue

        # when the destination is reached return
        if current == destination:
            return path, distance

        # gp through other neighbouring towns
        for neighbour, road_distance in roads.get(current, {}).items():

            # blocked roadsti be left out
            if (current, neighbour) in blocked_roads:
                continue

            new_distance = distance + road_distance

            # Update if a shorter route is found
            if new_distance < best_distance.get(neighbour, float("inf")):
                best_distance[neighbour] = new_distance
                heapq.heappush(
                    queue,
                    (new_distance, neighbour, path + [neighbour])
                )

    # No available routes
    return None, float("inf")

In [51]:
# STEP 13: TEST THE ALTERNATIVE ROUTE PLANNER

# Temporarily block the Nairobi to Naivasha road
blocked_roads.add(("Nairobi", "Naivasha"))

# Find an alternative route from Nairobi to Kisumu
route, distance = alternative_route(
    "Nairobi",
    "Kisumu",
    roads,
    blocked_roads
)

# Display the alternative route and its total distance
print("Alternative Route:", route)
print("Total Distance:", distance, "km")

# Remove the temporary blockage after testing
blocked_roads.discard(("Nairobi", "Naivasha"))

Alternative Route: ['Nairobi', 'Narok', 'Bomet', 'Litein', 'Kericho', 'Kisumu']
Total Distance: 384.1 km


In [52]:
# STEP 14: INTEGRATE THE INFERENCE ENGINE AND ROUTE PLANNER

def trailer_decision(current_location,next_location,destination,light,weather):

    #inference engine fetermines decisions
    decision = inference_engine(current_location,next_location,light,weather)

    # If the road is blocked, search for an alternative route
    if decision.startswith("FIND"):

        print("Blocked road detected!")
        print("Searching for an alternative route...")

        # the shirterst route is found using ucs
        route, distance = alternative_route(
            current_location,
            destination,
            roads,
            blocked_roads
        )

        # deos another alternative route exist?
        if route is None:
            return "No alternative route available."

        # Return the alternative route and its distance
        return {
            "decision": decision,
            "alternative_route": route,
            "distance_km": distance
        }

    # If the road is not blocked, return the inference-engine decision
    return {
        "decision": decision
    }

In [53]:
# STEP 15: TEST THE INTEGRATED SYSTEM

# block the Nairobi to Naivasha road for a bit
blocked_roads.add(("Nairobi", "Naivasha"))

# Run the integrated trailer decision system
result = trailer_decision(
    "Nairobi",
    "Naivasha",
    "Kisumu",
    "Green",
    "Clear"
)

#final system output
print("Final Autonomous Trailer Decision:")
print(result)

# temporary blockage after testing removed
blocked_roads.discard(("Nairobi", "Naivasha"))

Blocked road detected!
Searching for an alternative route...
Final Autonomous Trailer Decision:
{'decision': 'FIND AN ALTERNATIVE ROUTE!!', 'alternative_route': ['Nairobi', 'Narok', 'Bomet', 'Litein', 'Kericho', 'Kisumu'], 'distance_km': 384.1}


# STEP 16: INTERACTIVE USER RUN (User Interaction)

In [54]:
def run_interactive():
    current = input("Current town: ").strip()
    nxt = input("Next town: ").strip()
    light = input("Traffic light (Red/Yellow/Green): ")
    weather = input("Weather (Clear/Rain/HeavyRain): ")
    blocked = input("Is the road ahead blocked? (y/n): ").strip().lower()

    if nxt not in roads.get(current, {}):
      print(f"No road from {current} to {nxt} in the knowledge base.")
    return

    if blocked == "y":
        block_road(current, nxt)

    print(trailer_decision(current, nxt, "Kisumu", light, weather))
    blocked_roads.discard((current, nxt))

run_interactive()

Current town: Naivasha
Next town: Nairobi
Traffic light (Red/Yellow/Green): Red
Weather (Clear/Rain/HeavyRain): Rain
Is the road ahead blocked? (y/n): y
No road from Naivasha to Nairobi in the knowledge base.
